# Imports

In [110]:
import pandas as pd
import numpy as np
import igraph as ig


# Part 1

In [140]:
df = pd.read_csv("data/Part_A/1/balanced_graph.csv")
g = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
g.es["sign"] = df["sign"].tolist()
g.simplify(combine_edges="first")
triangles = g.cliques(min=3, max=3)

In [142]:
zero_edges = [e.index for e in g.es if e["sign"] == 0]
triangle_edges = []
for tri in triangles:
    n1, n2, n3 = tri
    e1 = g.get_eid(n1, n2)
    e2 = g.get_eid(n2, n3)
    e3 = g.get_eid(n3, n1)
    triangle_edges.append((e1, e2, e3))



In [143]:
def is_triangle_balanced(tri_edges, graph):
    prod = 1
    for e in tri_edges:
        s = graph.es[e]["sign"]
        if s == 0:
            return True  # unknown edges can still be assigned
        prod *= s
    return prod > 0


In [144]:
def backtrack_balance(graph, zero_edges, triangle_edges, idx=0):
    if idx == len(zero_edges):
        for tri in triangle_edges:
            if not is_triangle_balanced(tri, graph):
                return False
        return True

    edge_idx = zero_edges[idx]

    for sign in [1, -1]:        
        graph.es[edge_idx]["sign"] = sign        
        valid = True
        for tri in triangle_edges:
            if edge_idx in tri and not is_triangle_balanced(tri, graph):
                valid = False
                break
        if valid:
            if backtrack_balance(graph, zero_edges, triangle_edges, idx + 1):
                return True        
        graph.es[edge_idx]["sign"] = 0
    return False


In [146]:
success = backtrack_balance(g, zero_edges, triangle_edges)
if success:
    print("Graph successfully balanced!")
else:
    print("No assignment can fully balance the graph with given constraints.")


Graph successfully balanced!
